<a href="https://colab.research.google.com/github/olawaleaboderin/AVCAD/blob/main/climate_data_annotated.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Climate Data Extraction — NASA POWER Agroclimatology Archive

**Project:** Climate Variability and Yield Stability of Major Staple Crops in Sub-Saharan Africa  
**Course:** Analysis and Visualisation of Complex Agro-Environmental Data  
**Programme:** Master's in Green Data Science, Instituto Superior de Agronomia, Universidade de Lisboa  

---

## Purpose of This Notebook

This notebook is a **data preparation script**. It is not part of the main analysis.  
Its sole purpose is to pull raw monthly climate data from NASA's POWER API for the five study countries and aggregate it to annual values, producing the climate half of the merged dataset used in `Climate_Yield_Stability_SSA.ipynb`.

The output files produced here are:
- `NASA_POWER_climate_annual.csv` — one row per Country × Year, used for the climate-yield merge
- `NASA_POWER_climate_monthly.csv` — one row per Country × Year × Month, retained for reference

---

## Data Source

**NASA POWER (Prediction Of Worldwide Energy Resources)**  
Agroclimatology Community archive, accessed via the POWER REST API.  
URL: https://power.larc.nasa.gov/  

Three climate parameters are extracted:

| API Parameter | Description | Unit |
|---|---|---|
| `PRECTOTCORR` | Bias-corrected precipitation | mm/day |
| `T2M` | Temperature at 2 metres | °C |
| `ALLSKY_SFC_SW_DWN` | All-sky surface shortwave downward irradiance | MJ/m²/day |

---


## Section 1 — Location Selection

> **AI prompt used:** *"For a climate-yield analysis covering Nigeria, Ghana, Kenya, Mali, and Ethiopia, suggest the most representative single latitude/longitude point for each country's main staple-crop (maize, millet, cassava) growing region, suitable for extracting NASA POWER climate data as a national proxy."*

| Country | Coordinate | Region represented |
|---|---|---|
| Nigeria | 10.5°N, 7.4°E | Guinea Savanna belt (Kaduna/Minna area) |
| Ghana | 9.4°N, 0.84°W | Northern Ghana (Tamale area) |
| Kenya | 0.5°N, 35.3°E | Rift Valley maize belt (Eldoret area) |
| Mali | 11.3°N, 5.7°W | Sikasso agricultural region |
| Ethiopia | 8.5°N, 37.1°E | Oromia maize belt (Jimma/Bako area) |

## Section 2 — API Call and Data Pull

> **AI prompt used:** *"Write a Python script using the requests library to pull monthly climate data (rainfall, temperature, solar radiation) from the NASA POWER REST API for five African country coordinates, looping over each country, handling API errors gracefully, skipping the month=13 annual summary entries that POWER includes in its JSON response, and storing all records in a flat list for conversion to a pandas DataFrame."*



In [ ]:
# ============================================================
# Section 2 — API Call and Data Pull
# ============================================================
# No installation required beyond the standard Colab environment.
# 'requests' is pre-installed in Google Colab.

import requests
import pandas as pd
import time

# ------------------------------------------------------------
# Representative coordinates for each country's main
# staple-crop growing region (see Section 1 above for rationale)
# ------------------------------------------------------------
locations = {
    "Nigeria":  {"lat": 10.5, "lon": 7.4},   # Guinea Savanna belt (Kaduna/Minna)
    "Ghana":    {"lat": 9.4,  "lon": -0.84}, # Northern Ghana (Tamale)
    "Kenya":    {"lat": 0.5,  "lon": 35.3},  # Rift Valley maize belt (Eldoret)
    "Mali":     {"lat": 11.3, "lon": -5.7},  # Sikasso region
    "Ethiopia": {"lat": 8.5,  "lon": 37.1},  # Oromia maize belt (Jimma/Bako)
}

# ------------------------------------------------------------
# NASA POWER API parameters
# PRECTOTCORR = bias-corrected precipitation (mm/day)
# T2M         = temperature at 2 metres (°C)
# ALLSKY_SFC_SW_DWN = solar radiation (MJ/m²/day)
# Community 'AG' = Agroclimatology (crop-relevant archive)
# ------------------------------------------------------------
parameters = "PRECTOTCORR,T2M,ALLSKY_SFC_SW_DWN"
base_url = "https://power.larc.nasa.gov/api/temporal/monthly/point"

# Solar radiation is only available from 1984 onward in NASA POWER.
# Setting this as the earliest common start year for all three parameters.
MIN_START_YEAR = 1984

all_records = []  # flat list — one entry per Country/Year/Month/Parameter

for country, coords in locations.items():
    # Ethiopia's national agricultural statistics only begin from 1993
    # (following the 1991 political transition and statistical reorganisation),
    # so there is no point pulling climate data earlier than 1993 for Ethiopia.
    start_year = max(MIN_START_YEAR, 1993 if country == "Ethiopia" else 1961)
    end_year = 2024

    # Build the API request URL for this country
    url = (
        f"{base_url}?parameters={parameters}&community=AG"
        f"&longitude={coords['lon']}&latitude={coords['lat']}"
        f"&start={start_year}&end={end_year}&format=JSON"
    )

    print(f"Fetching {country} ({start_year}-{end_year})...")
    resp = requests.get(url)

    # Handle HTTP-level errors (e.g. server down, bad coordinates)
    if resp.status_code != 200:
        print(f"  ERROR: HTTP {resp.status_code} for {country}")
        print(f"  Response: {resp.text[:500]}")
        continue

    data = resp.json()

    # Handle unexpected JSON structure (API schema may change)
    if "properties" not in data:
        print(f"  ERROR: Unexpected response structure for {country}")
        print(f"  Full response: {data}")
        continue

    params_data = data["properties"]["parameter"]

    for param_name, monthly_values in params_data.items():
        for yyyymm, value in monthly_values.items():
            year = int(yyyymm[:4])
            month = yyyymm[4:6]
            # NASA POWER includes a month=13 entry per year as an annual summary.
            # We skip these here and compute our own annual aggregates below.
            if month == "13":
                continue
            all_records.append({
                "Country": country,
                "Year": year,
                "Month": int(month),
                "Parameter": param_name,
                "Value": value
            })

    print(f"  OK — {len(monthly_values)} months pulled per parameter")
    time.sleep(1)  # polite pause between API calls

# Safety check — abort early if nothing was retrieved
if not all_records:
    raise RuntimeError("No data was successfully pulled. Check error messages above.")

print(f"\nTotal records collected (Country × Year × Month × Parameter): {len(all_records)}")

## Section 3 — Reshape to Wide Format

> **AI prompt used:** *"Convert the flat long-format records list (Country, Year, Month, Parameter, Value) into a wide-format DataFrame with one row per Country × Year × Month and one column per climate parameter, using pandas pivot_table, then rename columns to human-readable names."*


In [ ]:
# ============================================================
# Section 3 — Reshape from Long to Wide Format
# ============================================================

df_long = pd.DataFrame(all_records)

# Pivot: rows = Country × Year × Month, columns = the three climate parameters
df_wide = df_long.pivot_table(
    index=["Country", "Year", "Month"],
    columns="Parameter",
    values="Value"
).reset_index()

# Rename API parameter codes to descriptive column names
df_wide = df_wide.rename(columns={
    "PRECTOTCORR":        "Rainfall_mm_day",       # daily precipitation (mm)
    "T2M":                "Temp_C",                 # temperature at 2m (°C)
    "ALLSKY_SFC_SW_DWN":  "Solar_rad_MJ_m2_day"    # solar radiation (MJ/m²/day)
})

print("Monthly wide-format DataFrame shape:", df_wide.shape)
print(df_wide.head(12))  # preview first 12 months (one year) for one country

## Section 4 — Aggregate to Annual Values

> **AI prompt used:** *"Aggregate the monthly climate DataFrame to annual values: sum monthly rainfall (converting from mm/day to mm/month using a 30.44-day month approximation before summing), take the mean of monthly temperature and solar radiation across the 12 months of each year, grouped by Country and Year."*


In [ ]:
# ============================================================
# Section 4 — Aggregate Monthly Data to Annual Values
# ============================================================

df_annual = df_wide.groupby(["Country", "Year"]).agg(
    # Rainfall: NASA POWER gives mm/day; multiply by 30.44 to get mm/month,
    # then sum across 12 months to get total annual rainfall in mm/year.
    Rainfall_mm_year=("Rainfall_mm_day", lambda x: (x * 30.44).sum()),

    # Temperature: simple mean of monthly mean temperatures across the year.
    Temp_C_mean=("Temp_C", "mean"),

    # Solar radiation: mean of monthly mean daily irradiance across the year.
    Solar_rad_MJ_m2_day_mean=("Solar_rad_MJ_m2_day", "mean")
).reset_index()

print("Annual climate DataFrame shape:", df_annual.shape)
print("\nSample output (first 10 rows):")
print(df_annual.head(10).round(3))

print("\nRecord count per country (should match expected year range):")
print(df_annual.groupby('Country')['Year'].agg(['count', 'min', 'max']))

## Section 5 — Quality Check

> **AI prompt used:** *"Run a quick quality check on the aggregated annual climate DataFrame: check for missing values, flag any rainfall or temperature values that look implausible for Sub-Saharan African agricultural regions, and print a summary statistics table by country."*


In [ ]:
# ============================================================
# Section 5 — Quality Check
# ============================================================

print("=== Missing values ===")
print(df_annual.isna().sum())

print("\n=== Annual climate summary statistics by country ===")
print(
    df_annual.groupby('Country')[
        ['Rainfall_mm_year', 'Temp_C_mean', 'Solar_rad_MJ_m2_day_mean']
    ].describe().round(2)
)

# Flag any rainfall values that exceed 4000 mm/year as potentially anomalous
# (serves as a first-pass plausibility check; genuine anomalies should be
# verified against independent literature before excluding)
high_rain = df_annual[df_annual['Rainfall_mm_year'] > 4000]
if not high_rain.empty:
    print("\n=== Rainfall values exceeding 4000 mm/year (verify against literature) ===")
    print(high_rain[['Country', 'Year', 'Rainfall_mm_year']])
else:
    print("\nNo rainfall values exceed 4000 mm/year — plausibility check passed.")

## Section 6 — Save Output Files

> **AI prompt used:** *"Save the annual and monthly aggregated climate DataFrames to CSV files in the working directory, and in a Google Colab context trigger an automatic download of the annual file so it can be merged with the FAOSTAT yield data in Excel or in the main analysis notebook."*


In [ ]:
# ============================================================
# Section 6 — Save Output Files
# ============================================================

# Save both files to the Colab working directory
df_annual.to_csv("NASA_POWER_climate_annual.csv", index=False)
df_wide.to_csv("NASA_POWER_climate_monthly.csv", index=False)

print("Files saved:")
print("  NASA_POWER_climate_annual.csv  — use this for the climate-yield merge")
print("  NASA_POWER_climate_monthly.csv — full monthly series retained for reference")

# Trigger automatic download of the annual file in Google Colab
# (Comment this out if running locally — the file is already in your working directory)
try:
    from google.colab import files
    files.download("NASA_POWER_climate_annual.csv")
    print("\nDownload triggered for NASA_POWER_climate_annual.csv")
except ImportError:
    # Not running in Colab — file is already saved locally
    print("\nNot running in Colab — file saved to working directory.")

print("\n=== Final annual dataset preview ===")
print(df_annual.round(3).to_string(index=False))